# A FAIR² Mental Health Survey Dataset from Kilifi County, Kenya: Demonstrating AI-Ready Data Standards for Data Infrastructure in Africa Exploration with `mlcroissant`
This notebook demonstrates how to load and explore a dataset defined by a Croissant schema using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the Croissant schema URL
croissant_url = "https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json"

# Load Dataset object
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

# Print metadata overview
print(f"{metadata.name}: {metadata.description}")
print(f"Published: {metadata.datePublished}, Keywords: {metadata.keywords}")
print(f"License: {metadata.license}")

## 2. Data Overview
Review available record sets, their `@id`, and the fields and their `@id`. The following code programmatically inspects the Croissant metadata to list all record sets, and the contained fields/columns.


In [ ]:
from pprint import pprint

record_set_ids = [rset['@id'] for rset in metadata.to_json().get('recordSet', [])]
print("Available Record Sets (@id):")
for rset in metadata.to_json().get('recordSet', []):
    print(f"- {rset['@id']} : {rset.get('name', '')}")

if not record_set_ids:
    print("No record sets embedded directly in the metadata. Attempting to infer from `distribution` files...")

# Some datasets list record sets within `distribution` entries
distribution_objects = metadata.to_json().get('distribution', [])
for dist in distribution_objects:
    if isinstance(dist, dict) and 'encodingFormat' in dist:
        print(f"Distribution File: {dist.get('contentUrl')} (format: {dist.get('encodingFormat')}) (@id {dist.get('@id', None)})")

# In mlcroissant, .record_sets gives list of RecordSet objects
print("\nRecord sets and their fields as discovered by mlcroissant:")
# This part uses mlcroissant API directly
for rset in dataset.record_sets:
    print(f"- Record Set name: {rset.name}")
    print(f"  @id: {rset.id}")
    print("  Fields:")
    for field in rset.fields:
        print(f"    - {field.name} (@id: {field.id}, dataType: {field.data_type})")

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. All references are made using `@id` fields as recommended by the Croissant standard and as reported in the overview.

In [ ]:
# List all record set ids (from section above)
record_sets = [rset.id for rset in dataset.record_sets]

print("Extracting DataFrames for each record set...\n")
dataframes = {}
for record_set_id in record_sets:
    records = list(dataset.records(record_set=record_set_id))
    df = pd.DataFrame(records)
    dataframes[record_set_id] = df
    print(f"Columns for RecordSet '@id'={record_set_id}:")
    print(df.columns.tolist())
    print(df.head(), "\n")

# For demonstration, let's choose the first record set (if exists)
if record_sets:
    main_record_set_id = record_sets[0]
    df_main = dataframes[main_record_set_id]
    print(f"Using record set '@id': {main_record_set_id}")
else:
    raise ValueError("No record sets found in this dataset.")


## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering by value, normalizing numeric fields, or grouping by categorical variables. All fields referenced by their `@id` (column names as loaded, typically matching the field `@id`).

In [ ]:
# Inspect which numeric fields are available
df = df_main

print("Available columns in main record set:")
print(df.columns.tolist())

# Example: Suppose there's a GAD-7 score field and Age as fields
# We'll pick a numeric field for EDA. If you want, choose e.g. 'gad7_score', 'phq9_score', or field with similar name.
possible_numeric_fields = [col for col in df.columns if df[col].dtype in ['int64', 'float64'] or pd.api.types.is_numeric_dtype(df[col])]
print("Numeric fields detected:", possible_numeric_fields)

if possible_numeric_fields:
    numeric_field = possible_numeric_fields[0]
    print(f"Using numeric field: {numeric_field}")
else:
    raise ValueError("No numeric field detected in available columns.")

# Set threshold for filtering (10 is reasonable for GAD-7, PHQ-9, PSQ)
threshold = 10
if numeric_field in df.columns:
    filtered_df = df[df[numeric_field] > threshold].copy()
    print(f"Filtered records with {numeric_field} > {threshold}:")
    print(filtered_df.head())
    # Normalize the field
    mean = filtered_df[numeric_field].mean()
    std = filtered_df[numeric_field].std()
    filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - mean) / std
    print(f"\nNormalized {numeric_field} for filtered records:")
    print(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

    # Categorize/group by another field if available (e.g. sex, gender, education)
    possible_group_fields = [col for col in df.columns if df[col].dtype=='object' and col!=numeric_field]
    group_field = None
    
    for candidate in ['gender', 'sex', 'sex_at_birth', 'marital_status', 'village', 'level_of_education', 'education_level', 'education']:
        if candidate in df.columns:
            group_field = candidate
            break
    if not group_field and possible_group_fields:
        group_field = possible_group_fields[0]

    if group_field:
        print(f"\nGrouping filtered data by '{group_field}':")
        grouped_df = filtered_df.groupby(group_field).mean(numeric_only=True)
        print(grouped_df.head())
else:
    print(f"Numeric field {numeric_field} not found in columns.")

## 5. Visualization
Visualize the distribution of the selected numeric field and compare it across categories (if available).

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Histogram of the numeric field
plt.figure(figsize=(7,4))
sns.histplot(df[numeric_field].dropna(), bins=15, kde=True)
plt.xlabel(numeric_field)
plt.title(f"Distribution of {numeric_field}")
plt.show()

# Boxplot by group_field if available
if 'group_field' in locals() and group_field:
    plt.figure(figsize=(8,5))
    sns.boxplot(x=group_field, y=numeric_field, data=df)
    plt.title(f"{numeric_field} by {group_field}")
    plt.xticks(rotation=45)
    plt.show()

## 6. Conclusion
In this notebook, we've demonstrated a full pipeline for dataset exploration using `mlcroissant`:
* Accessed Croissant metadata and discovered available record sets and fields by their `@id`
* Loaded records using standard identifiers and extracted them into DataFrames
* Performed filtering, normalization, and grouping (aggregation) on a selected numeric field
* Visualized the data distribution and compared values across categories

This AI-ready dataset enabled quick discovery and profiling of mental health survey results in Kilifi County, supporting further analysis or modeling.